## Parsing AAER Cases

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [ ]:
resp = requests.get('https://www.sec.gov/enforcement-litigation/accounting-auditing-enforcement-releases', headers={"User-Agent": "AFDEN-class-demo (leye.li@unsw.edu.au)"})
soup = BeautifulSoup(resp.text, "html.parser")

### 1. Find the table needed

In [ ]:
rows = soup.find('table', class_='usa-table views-table views-view-table cols-2').find('tbody').find_all('tr')

In [ ]:
rows[0]

In [ ]:
rows[0].find_all('td')[0]

In [ ]:
rows[0].find_all('td')[1].get_text(strip=True)

### 2. Turn the texts in the table into a Pandas Dataframe

In [ ]:
data = []
for row in rows:
    cols = row.find_all('td')
    cols = [col.get_text(strip=True) for col in cols]
    data.append(cols)

df = pd.DataFrame(data, columns=['Date', 'Respondents'])
df.head()

### 3. Further clean the data

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed')

In [ ]:
text = 'Timothy Daly, CPARelease No.34-103008, AAER-4568'
re.findall(r'Release No\.(.+\, AAER-[0-9]+)', text)

In [ ]:
df['defendants'] = df['Respondents'].apply(lambda x: x.split('Release No.')[0].split('; '))

In [ ]:
df['Case_No'] = df['Respondents'].apply(
	lambda x: re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x)[0] if re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x) else None
)

### Aggregate all the previous steps into one function

In [ ]:
def Extract_AAER(page_num: int, 
                 base_url : str = 'https://www.sec.gov/enforcement-litigation/accounting-auditing-enforcement-releases?page='
                 ) -> pd.DataFrame:
    resp = requests.get(f'{base_url}{page_num}', 
                        headers={"User-Agent": "AFDEN-class-demo (leye.li@unsw.edu.au)"})
    soup = BeautifulSoup(resp.text, "html.parser")

    rows = soup.find('table', class_='usa-table views-table views-view-table cols-2').find('tbody').find_all('tr')
    
    data = []
    for row in rows:
        cols = row.find_all('td')
        cols = [col.get_text(strip=True) for col in cols]
        data.append(cols)
    
    df = pd.DataFrame(data, columns=['Date', 'temp'])
    df['Date'] = pd.to_datetime(df['Date'], format='mixed')
    df['defendants'] = df['temp'].apply(lambda x: x.split('Release No.')[0].split('; '))
    df['Case_No'] = df['temp'].apply(
	    lambda x: re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x)[0] if re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x) else None
        )
    df = df.drop(columns=['temp'])
    return df

In [ ]:
Extract_AAER(8)